# Train 8-Fold MobileNetV3-Small

Run the official 8-fold cross-rotation MobileNetV3-small experiments and write per-fold metrics, predictions, and figures.

In [12]:
from __future__ import annotations

from meatlens_pork_pipeline.notebook_progress import (
    advance_notebook_cell_progress,
    finish_notebook_cell_progress,
    iter_notebook_progress,
    start_notebook_cell_progress,
)
NB_04_TRAIN_8FOLD_MOBILENETV3SMALL_CELL_PROGRESS_1 = start_notebook_cell_progress('04_train_8fold_mobilenetv3small.ipynb', 'Load training dependencies', total_steps=1)

import json
import os
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support
from sklearn.utils.class_weight import compute_class_weight
from meatlens_pork_pipeline.augmentation import build_training_augmentation
from meatlens_pork_pipeline.evaluation import evaluate_model as pipeline_evaluate_model
from meatlens_pork_pipeline.training import train_model as pipeline_train_model

shared_notebook = json.loads(Path('00_shared_setup.ipynb').read_text(encoding='utf-8'))
shared_code = '\n\n'.join(
    ''.join(cell.get('source', []))
    for cell in shared_notebook['cells']
    if cell.get('cell_type') == 'code'
)
exec(shared_code, globals())

mplconfig_dir = ensure_dir(ROOT / '.matplotlib')
os.environ.setdefault('MPLCONFIGDIR', str(mplconfig_dir.resolve()))

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import callbacks, layers
from tensorflow.keras.applications import MobileNetV3Small
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input

finish_notebook_cell_progress(NB_04_TRAIN_8FOLD_MOBILENETV3SMALL_CELL_PROGRESS_1)


[START] 04_train_8fold_mobilenetv3small.ipynb | Load training dependencies [0/1 step] elapsed=0.0s
[START] 00_shared_setup.ipynb | Shared setup bootstrap [0/1 step] elapsed=0.0s
Default processed dataset not ready at C:\Users\Adriaan M. Dimate\Desktop\development\school\meatlens-training-2\data\roboflow_processed_hsv_lab_threshold_roi_224 - run the early notebooks with raw or Excel input, or set overrides.
[RUNNING] 00_shared_setup.ipynb | Shared setup bootstrap | done [0/1 step] elapsed=0.0s
[RUNNING] 00_shared_setup.ipynb | Shared setup bootstrap | done [1/1 step] elapsed=0.0s
[RUNNING] 04_train_8fold_mobilenetv3small.ipynb | Load training dependencies | done [0/1 step] elapsed=0.0s
[RUNNING] 04_train_8fold_mobilenetv3small.ipynb | Load training dependencies | done [1/1 step] elapsed=0.0s


In [13]:
from meatlens_pork_pipeline.notebook_progress import (
    advance_notebook_cell_progress,
    finish_notebook_cell_progress,
    iter_notebook_progress,
    start_notebook_cell_progress,
)
NB_04_TRAIN_8FOLD_MOBILENETV3SMALL_CELL_PROGRESS_2 = start_notebook_cell_progress('04_train_8fold_mobilenetv3small.ipynb', 'Define training helpers', total_steps=1)

def configure_tensorflow_for_training() -> list[str]:
    gpu_names = enforce_training_gpu()
    for gpu in tf.config.list_physical_devices('GPU'):
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except Exception:
            pass
    return gpu_names


TRAINING_AUGMENTATION = build_training_augmentation(AUGMENTATION_PRESET)


class ImageOnlySequence(tf.keras.utils.Sequence):
    def __init__(
        self,
        df: pd.DataFrame,
        batch_size: int,
        shuffle: bool = False,
        use_augmentation: bool = False,
    ) -> None:
        super().__init__()
        self.df = df.reset_index(drop=True).copy()
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.use_augmentation = use_augmentation
        self.indexes = np.arange(len(self.df))
        self.on_epoch_end()

    def __len__(self) -> int:
        return int(np.ceil(len(self.df) / self.batch_size))

    def __getitem__(self, index: int) -> tuple[np.ndarray, np.ndarray]:
        batch_indexes = self.indexes[index * self.batch_size : (index + 1) * self.batch_size]
        batch_df = self.df.iloc[batch_indexes]
        images: list[np.ndarray] = []
        labels: list[int] = []
        for row in batch_df.to_dict(orient='records'):
            image = Image.open(Path(str(row['local_image_path']))).convert('RGB')
            if image.size != TARGET_SIZE:
                image = image.resize(TARGET_SIZE, Image.BILINEAR)
            image_uint8 = np.asarray(image, dtype=np.uint8)
            images.append(image_uint8.astype(np.float32))
            labels.append(LABEL_ORDER.index(str(row['label'])))
        batch = np.stack(images, axis=0).astype(np.float32)
        if self.use_augmentation:
            batch = TRAINING_AUGMENTATION(batch, training=True).numpy()
        batch = preprocess_input(batch)
        return batch, tf.keras.utils.to_categorical(labels, num_classes=len(LABEL_ORDER))

    def on_epoch_end(self) -> None:
        if self.shuffle:
            np.random.shuffle(self.indexes)


def build_mobilenetv3small_cnn_only_model(weights: str | None = 'imagenet') -> tf.keras.Model:
    backbone = MobileNetV3Small(
        include_top=False,
        include_preprocessing=False,
        input_shape=INPUT_SHAPE,
        weights=weights,
        alpha=1.0,
    )
    backbone._name = 'mobilenetv3small_backbone'
    backbone.trainable = False

    inputs = tf.keras.Input(shape=INPUT_SHAPE, name='image_input')
    x = backbone(inputs, training=False)
    x = layers.GlobalAveragePooling2D(name='avg_pool')(x)
    x = layers.Dropout(0.2, name='dropout_1')(x)
    x = layers.Dense(128, activation='relu', name='dense_128')(x)
    x = layers.Dropout(0.1, name='dropout_2')(x)
    outputs = layers.Dense(len(LABEL_ORDER), activation='softmax', name='predictions')(x)
    model = tf.keras.Model(inputs=inputs, outputs=outputs, name='meatlens_mobilenetv3small_cnn_only')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=HEAD_LR),
        loss='categorical_crossentropy',
        metrics=['accuracy'],
    )
    return model


def compute_class_weights_from_labels(labels: np.ndarray) -> dict[int, float]:
    label_indices = np.array([LABEL_ORDER.index(str(label)) for label in labels], dtype=int)
    weights = compute_class_weight(
        class_weight='balanced',
        classes=np.arange(len(LABEL_ORDER), dtype=int),
        y=label_indices,
    )
    return {int(index): float(weight) for index, weight in enumerate(weights)}


def find_backbone_layer(model: tf.keras.Model) -> tf.keras.Model:
    for layer in model.layers:
        if isinstance(layer, tf.keras.Model):
            return layer
    raise ValueError('Expected a nested backbone model in the classifier.')


class ValF1Callback(callbacks.Callback):
    def __init__(self, validation_sequence: ImageOnlySequence) -> None:
        super().__init__()
        self.validation_sequence = validation_sequence

    def on_epoch_end(self, epoch: int, logs: dict[str, float] | None = None) -> None:
        logs = logs or {}
        probabilities = self.model.predict(self.validation_sequence, verbose=0)
        y_pred = probabilities.argmax(axis=1)
        y_true = self.validation_sequence.df['label'].map(LABEL_ORDER.index).to_numpy(dtype=int)
        _, _, macro_f1, _ = precision_recall_fscore_support(
            y_true,
            y_pred,
            labels=np.arange(len(LABEL_ORDER), dtype=int),
            average='macro',
            zero_division=0,
        )
        logs['val_f1_macro'] = float(macro_f1)


def save_confusion_matrix_png(confusion: np.ndarray, output_path: Path) -> None:
    figure, axis = plt.subplots(figsize=(6, 5))
    sns.heatmap(
        confusion,
        annot=True,
        fmt='.0f',
        cmap='Blues',
        xticklabels=LABEL_ORDER,
        yticklabels=LABEL_ORDER,
        ax=axis,
    )
    axis.set_xlabel('Predicted')
    axis.set_ylabel('Actual')
    figure.tight_layout()
    figure.savefig(output_path, dpi=200)
    plt.close(figure)


def compute_severe_error_rate(y_true: pd.Series, y_pred: pd.Series) -> float:
    total = len(y_true)
    if total == 0:
        return 0.0
    severe = 0
    for true_label, pred_label in zip(y_true.astype(str), y_pred.astype(str), strict=True):
        if (true_label, pred_label) in SEVERE_ERROR_LABEL_PAIRS:
            severe += 1
    return float(severe / total)


def summarize_fold_metrics(prediction_df: pd.DataFrame) -> dict[str, float]:
    y_true = prediction_df['true_label'].astype(str)
    y_pred = prediction_df['predicted_label'].astype(str)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=LABEL_ORDER,
        average='macro',
        zero_division=0,
    )
    return {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'macro_precision': float(precision),
        'macro_recall': float(recall),
        'macro_f1': float(f1),
        'severe_error_rate': compute_severe_error_rate(y_true, y_pred),
    }


def evaluate_predictions(model: tf.keras.Model, test_df: pd.DataFrame, batch_size: int) -> tuple[pd.DataFrame, dict[str, float], np.ndarray]:
    test_sequence = ImageOnlySequence(test_df, batch_size=batch_size, shuffle=False, use_augmentation=False)
    probabilities = model.predict(test_sequence, verbose=0)
    predicted_indices = probabilities.argmax(axis=1)
    prediction_rows: list[dict[str, object]] = []
    for row, predicted_index, probability_vector in zip(
        test_df.to_dict(orient='records'),
        predicted_indices,
        probabilities,
        strict=True,
    ):
        prediction_rows.append(
            {
                **row,
                'true_label': str(row['label']),
                'predicted_label': LABEL_ORDER[int(predicted_index)],
                'confidence': float(probability_vector[int(predicted_index)]),
                'fresh_probability': float(probability_vector[0]),
                'not_fresh_probability': float(probability_vector[1]),
                'spoiled_probability': float(probability_vector[2]),
            }
        )
    prediction_df = pd.DataFrame(prediction_rows)
    y_true = prediction_df['true_label'].astype(str)
    y_pred = prediction_df['predicted_label'].astype(str)
    confusion = confusion_matrix(y_true, y_pred, labels=LABEL_ORDER)
    metrics = summarize_fold_metrics(prediction_df)
    return prediction_df, metrics, confusion

finish_notebook_cell_progress(NB_04_TRAIN_8FOLD_MOBILENETV3SMALL_CELL_PROGRESS_2)


[START] 04_train_8fold_mobilenetv3small.ipynb | Define training helpers [0/1 step] elapsed=0.0s
[RUNNING] 04_train_8fold_mobilenetv3small.ipynb | Define training helpers | done [0/1 step] elapsed=0.0s
[RUNNING] 04_train_8fold_mobilenetv3small.ipynb | Define training helpers | done [1/1 step] elapsed=0.0s


In [14]:
from meatlens_pork_pipeline.notebook_progress import (
    advance_notebook_cell_progress,
    finish_notebook_cell_progress,
    iter_notebook_progress,
    start_notebook_cell_progress,
)
NB_04_TRAIN_8FOLD_MOBILENETV3SMALL_CELL_PROGRESS_3 = start_notebook_cell_progress('04_train_8fold_mobilenetv3small.ipynb', 'Define fold training', total_steps=1)

def train_single_fold(
    fold_name: str,
    seed: int,
    train_csv: Path,
    val_csv: Path,
    test_csv: Path,
    output_root: Path,
    model_weights: str | None,
    batch_size: int,
    epochs_head: int,
    epochs_fine: int,
    use_training_augmentation: bool,
) -> dict[str, object]:
    set_global_seed(seed)
    train_df = pd.read_csv(train_csv, dtype=str).fillna('')
    val_df = pd.read_csv(val_csv, dtype=str).fillna('')
    test_df = pd.read_csv(test_csv, dtype=str).fillna('')

    models_dir = ensure_dir(output_root / 'models')
    histories_dir = ensure_dir(output_root / 'histories')
    predictions_dir = ensure_dir(output_root / 'predictions')
    figures_dir = ensure_dir(output_root / 'figures')

    run_slug = f'processed_roi8_cnn_only_{fold_name}_seed{seed}'
    checkpoint_path = models_dir / f'{run_slug}_best.keras'
    model_path = models_dir / f'{run_slug}.keras'
    history_path = histories_dir / f'{run_slug}_history.csv'
    predictions_path = predictions_dir / f'{run_slug}_test_predictions.csv'
    confusion_csv_path = figures_dir / f'{run_slug}_confusion_matrix.csv'
    confusion_png_path = figures_dir / f'{run_slug}_confusion_matrix.png'

    if TRAINING_STRATEGY != 'end_to_end':
        package_run_key = f'{fold_name}_s{seed}'
        package_model_dir = ensure_dir(output_root / '_p' / package_run_key)
        package_metrics_dir = ensure_dir(output_root / '_e' / package_run_key)
        training_artifacts = pipeline_train_model(
            train_csv=train_csv,
            val_csv=val_csv,
            output_dir=package_model_dir,
            seed=seed,
            epochs_head=epochs_head,
            epochs_fine=epochs_fine,
            head_lr=HEAD_LR,
            fine_tune_lr=FINE_TUNE_LR,
            training_strategy=TRAINING_STRATEGY,
            weights=model_weights,
            batch_size=batch_size,
            augmentation=use_training_augmentation,
            fine_tune_fraction=FINE_TUNE_FRACTION,
        )
        evaluation_summary = pipeline_evaluate_model(
            model_h5_path=training_artifacts.model_h5_path,
            test_csv=test_csv,
            output_dir=package_metrics_dir,
            batch_size=batch_size,
        )
        shutil.copy2(training_artifacts.model_h5_path, model_path)
        shutil.copy2(training_artifacts.history_csv_path, history_path)
        shutil.copy2(evaluation_summary.predictions_csv_path, predictions_path)
        shutil.copy2(evaluation_summary.confusion_matrix_csv_path, confusion_csv_path)
        shutil.copy2(evaluation_summary.confusion_matrix_png_path, confusion_png_path)
        prediction_df = pd.read_csv(predictions_path)
        metrics = dict(evaluation_summary.metrics)
        metrics['severe_error_rate'] = compute_severe_error_rate(
            prediction_df['true_label'],
            prediction_df['predicted_label'],
        )
        return {
            'fold': fold_name,
            'seed': seed,
            'dataset_source': DATASET_SOURCE,
            'training_strategy': TRAINING_STRATEGY,
            'input_mode': INPUT_MODE,
            'head_lr': float(HEAD_LR),
            'fine_tune_lr': float(FINE_TUNE_LR),
            'epochs_head': int(epochs_head),
            'epochs_fine': int(epochs_fine),
            'fine_tune_fraction': float(FINE_TUNE_FRACTION),
            'accuracy': metrics['accuracy'],
            'macro_precision': metrics['macro_precision'],
            'macro_recall': metrics['macro_recall'],
            'macro_f1': metrics['macro_f1'],
            'severe_error_rate': metrics['severe_error_rate'],
            'train_count': len(train_df),
            'val_count': len(val_df),
            'test_count': len(test_df),
            'model_path': str(model_path),
            'predictions_path': str(predictions_path),
            'confusion_matrix_csv': str(confusion_csv_path),
            'confusion_matrix_png': str(confusion_png_path),
        }

    train_sequence = ImageOnlySequence(
        train_df,
        batch_size=batch_size,
        shuffle=True,
        use_augmentation=use_training_augmentation,
    )
    val_sequence = ImageOnlySequence(val_df, batch_size=batch_size, shuffle=False, use_augmentation=False)
    class_weights = compute_class_weights_from_labels(train_df['label'].to_numpy())

    model = build_mobilenetv3small_cnn_only_model(weights=model_weights)
    val_f1_callback = ValF1Callback(val_sequence)
    fit_callbacks: list[callbacks.Callback] = [
        val_f1_callback,
        callbacks.ModelCheckpoint(
            filepath=str(checkpoint_path),
            monitor='val_f1_macro',
            mode='max',
            save_best_only=True,
            verbose=0,
        ),
        callbacks.EarlyStopping(
            monitor='val_f1_macro',
            mode='max',
            patience=3,
            restore_best_weights=True,
            verbose=0,
        ),
        callbacks.ReduceLROnPlateau(
            monitor='val_f1_macro',
            mode='max',
            factor=0.5,
            patience=1,
            min_lr=1e-6,
            verbose=0,
        ),
    ]

    histories: list[pd.DataFrame] = []
    if epochs_head > 0:
        head_history = model.fit(
            train_sequence,
            validation_data=val_sequence,
            epochs=epochs_head,
            class_weight=class_weights,
            callbacks=fit_callbacks,
            verbose=1,
        )
        head_df = pd.DataFrame(head_history.history)
        head_df['phase'] = 'head'
        head_df['epoch'] = np.arange(1, len(head_df) + 1)
        histories.append(head_df)

    if epochs_fine > 0 and FINE_TUNE_FRACTION > 0.0:
        backbone = find_backbone_layer(model)
        backbone.trainable = True
        if FINE_TUNE_FRACTION >= 1.0:
            for layer in backbone.layers:
                layer.trainable = True
        else:
            fine_tune_at = max(int(len(backbone.layers) * (1.0 - FINE_TUNE_FRACTION)), 1)
            for layer in backbone.layers[:fine_tune_at]:
                layer.trainable = False
            for layer in backbone.layers[fine_tune_at:]:
                layer.trainable = True

        model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=FINE_TUNE_LR),
            loss='categorical_crossentropy',
            metrics=['accuracy'],
        )
        fine_history = model.fit(
            train_sequence,
            validation_data=val_sequence,
            epochs=epochs_head + epochs_fine,
            initial_epoch=epochs_head,
            class_weight=class_weights,
            callbacks=fit_callbacks,
            verbose=1,
        )
        fine_df = pd.DataFrame(fine_history.history)
        fine_df['phase'] = 'fine_tune'
        fine_df['epoch'] = np.arange(epochs_head + 1, epochs_head + len(fine_df) + 1)
        histories.append(fine_df)

    if checkpoint_path.exists():
        model = tf.keras.models.load_model(checkpoint_path, compile=False)
        model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=FINE_TUNE_LR),
            loss='categorical_crossentropy',
            metrics=['accuracy'],
        )

    ensure_dir(model_path.parent)
    model.save(model_path, include_optimizer=False)
    history_df = pd.concat(histories, ignore_index=True) if histories else pd.DataFrame()
    ensure_dir(history_path.parent)
    history_df.to_csv(history_path, index=False)

    prediction_df, metrics, confusion = evaluate_predictions(model, test_df, batch_size=batch_size)
    ensure_dir(predictions_path.parent)
    prediction_df.to_csv(predictions_path, index=False)
    ensure_dir(confusion_csv_path.parent)
    pd.DataFrame(confusion, index=LABEL_ORDER, columns=LABEL_ORDER).to_csv(confusion_csv_path)
    save_confusion_matrix_png(confusion, confusion_png_path)

    return {
        'fold': fold_name,
        'seed': seed,
        'dataset_source': DATASET_SOURCE,
        'training_strategy': TRAINING_STRATEGY,
        'input_mode': INPUT_MODE,
        'head_lr': float(HEAD_LR),
        'fine_tune_lr': float(FINE_TUNE_LR),
        'epochs_head': int(epochs_head),
        'epochs_fine': int(epochs_fine),
        'fine_tune_fraction': float(FINE_TUNE_FRACTION),
        'accuracy': metrics['accuracy'],
        'macro_precision': metrics['macro_precision'],
        'macro_recall': metrics['macro_recall'],
        'macro_f1': metrics['macro_f1'],
        'severe_error_rate': metrics['severe_error_rate'],
        'train_count': len(train_df),
        'val_count': len(val_df),
        'test_count': len(test_df),
        'model_path': str(model_path),
        'predictions_path': str(predictions_path),
        'confusion_matrix_csv': str(confusion_csv_path),
        'confusion_matrix_png': str(confusion_png_path),
    }

finish_notebook_cell_progress(NB_04_TRAIN_8FOLD_MOBILENETV3SMALL_CELL_PROGRESS_3)


[START] 04_train_8fold_mobilenetv3small.ipynb | Define fold training [0/1 step] elapsed=0.0s
[RUNNING] 04_train_8fold_mobilenetv3small.ipynb | Define fold training | done [0/1 step] elapsed=0.0s
[RUNNING] 04_train_8fold_mobilenetv3small.ipynb | Define fold training | done [1/1 step] elapsed=0.0s


In [15]:
from meatlens_pork_pipeline.notebook_progress import (
    advance_notebook_cell_progress,
    finish_notebook_cell_progress,
    iter_notebook_progress,
    start_notebook_cell_progress,
)
NB_04_TRAIN_8FOLD_MOBILENETV3SMALL_CELL_PROGRESS_4 = start_notebook_cell_progress('04_train_8fold_mobilenetv3small.ipynb', 'Run 8-fold training', total_steps=1)

RUN_SEEDS_LOCAL = [int(seed) for seed in override('RUN_SEEDS', RUN_SEEDS)]
default_select_folds = [f'fold{i}' for i in range(1, 9)]
SELECT_FOLDS = [str(fold) for fold in override('SELECT_FOLDS', default_select_folds)]
MODEL_WEIGHTS = override('MODEL_WEIGHTS', 'imagenet')
FINE_TUNE_FRACTION = resolve_fine_tune_fraction(override('FINE_TUNE_FRACTION', FINE_TUNE_FRACTION))
TRAINING_STRATEGY = str(override('TRAINING_STRATEGY', TRAINING_STRATEGY))
USE_TRAINING_AUGMENTATION = bool(override('USE_TRAINING_AUGMENTATION', True))
CROSS_ROTATION_ROOT = Path(str(override('CROSS_ROTATION_ROOT', GENERATED_SPLITS_ROOT)))
BASE_ROBOFLOW_OUTPUT_ROOT = TRAINING_OUTPUTS_ROOT
if TRAINING_STRATEGY == 'training1_compatible_end_to_end':
    default_eightfold_output_root = BASE_ROBOFLOW_OUTPUT_ROOT / 'mobilenetv3small_8fold_processed_roi_cnn_only_training1_compatible_end_to_end'
elif TRAINING_STRATEGY == 'roboflow_cached_baseline_v1':
    default_eightfold_output_root = BASE_ROBOFLOW_OUTPUT_ROOT / 'mobilenetv3small_8fold_processed_roi_cnn_only'
else:
    raise ValueError(
        "TRAINING_STRATEGY must be 'training1_compatible_end_to_end' or 'roboflow_cached_baseline_v1' for the official Roboflow notebook run."
    )
OUTPUT_ROOT = ensure_dir(Path(str(override('EIGHTFOLD_OUTPUT_ROOT', default_eightfold_output_root))))
METRICS_CSV_PATH = OUTPUT_ROOT / 'processed_roi8_cnn_only_seed_metrics.csv'

detected_gpu_names = configure_tensorflow_for_training()
if detected_gpu_names:
    print(f'Using GPU(s): {detected_gpu_names}')

summary_rows: list[dict[str, object]] = []
run_pairs = [(seed, fold_name) for seed in RUN_SEEDS_LOCAL for fold_name in SELECT_FOLDS]
for seed, fold_name in iter_notebook_progress(
    run_pairs,
    '04_train_8fold_mobilenetv3small.ipynb | train fold runs',
    total=len(run_pairs),
    unit='run',
    leave=True,
):
    train_csv = CROSS_ROTATION_ROOT / f'{fold_name}_train.csv'
    val_csv = CROSS_ROTATION_ROOT / f'{fold_name}_val.csv'
    test_csv = CROSS_ROTATION_ROOT / f'{fold_name}_test.csv'
    if not train_csv.exists() or not val_csv.exists() or not test_csv.exists():
        raise FileNotFoundError(f'Missing split files for {fold_name} under {CROSS_ROTATION_ROOT}')
    summary_rows.append(
        train_single_fold(
            fold_name=fold_name,
            seed=seed,
            train_csv=train_csv,
            val_csv=val_csv,
            test_csv=test_csv,
            output_root=OUTPUT_ROOT,
            model_weights=MODEL_WEIGHTS,
            batch_size=BATCH_SIZE,
            epochs_head=EPOCHS_HEAD,
            epochs_fine=EPOCHS_FINE,
            use_training_augmentation=USE_TRAINING_AUGMENTATION,
        )
    )

metrics_df = pd.DataFrame(summary_rows)
metrics_df.to_csv(METRICS_CSV_PATH, index=False)
print(metrics_df[['fold', 'seed', 'accuracy', 'macro_f1']].to_string(index=False))

finish_notebook_cell_progress(NB_04_TRAIN_8FOLD_MOBILENETV3SMALL_CELL_PROGRESS_4)


[START] 04_train_8fold_mobilenetv3small.ipynb | Run 8-fold training [0/1 step] elapsed=0.0s
Using GPU(s): ['NVIDIA GeForce RTX 4050 Laptop GPU']
[START] 04_train_8fold_mobilenetv3small.ipynb | train fold runs [0/24 run] elapsed=0.0s
[RUNNING] 04_train_8fold_mobilenetv3small.ipynb | train fold runs [1/24 run] elapsed=35.2s
[RUNNING] 04_train_8fold_mobilenetv3small.ipynb | train fold runs [2/24 run] elapsed=162.9s
[RUNNING] 04_train_8fold_mobilenetv3small.ipynb | train fold runs [3/24 run] elapsed=237.2s
[RUNNING] 04_train_8fold_mobilenetv3small.ipynb | train fold runs [4/24 run] elapsed=320.0s
[RUNNING] 04_train_8fold_mobilenetv3small.ipynb | train fold runs [5/24 run] elapsed=403.3s
[RUNNING] 04_train_8fold_mobilenetv3small.ipynb | train fold runs [6/24 run] elapsed=484.1s
[RUNNING] 04_train_8fold_mobilenetv3small.ipynb | train fold runs [7/24 run] elapsed=558.8s
[RUNNING] 04_train_8fold_mobilenetv3small.ipynb | train fold runs [8/24 run] elapsed=634.5s
[RUNNING] 04_train_8fold_mobilen